# Compare the rebuilt SFINCS runs: maximum water depth and maximum water level

Compares `subgrid_200m` and `subgrid_400m` against the regular 100 m reference (`regular_100m`).

- **Maximum depth (hmax):** for every run, `zsmax` (maximum over the whole simulation) is downscaled onto the
  100 m DEM: `hmax = max(zsmax - dem_100m, 0)`. Each 200 m / 400 m cell covers an exact 2x2 / 4x4 block of DEM cells.
- **Maximum water level (zsmax):** compared on the same 100 m grid, only in cells flooded in the reference
  (hmax > 5 cm); elsewhere the regular zsmax is undefined (never wet).

Run the models first (`models/rebuild/run_sfincs_all.bat`, see `scripts/sfincs_rebuild/write_batch.py`).


### Imports and paths

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[1]  # notebooks/sfincs_rebuild -> repository root
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.sfincs_rebuild.compare import compare_all, plot_case, summarize

MODEL_ROOT = PROJECT_ROOT / "models" / "rebuild"
OUTPUT_FOLDER = PROJECT_ROOT / "outputs" / "rebuild"
case_table = pd.read_csv(OUTPUT_FOLDER / "case_table.csv")

## Metrics per case

In [ ]:
comparison = compare_all(case_table, PROJECT_ROOT, MODEL_ROOT, out_csv=OUTPUT_FOLDER / "comparison.csv")
print(comparison["status"].value_counts())
comparison.head()

## Summary per split and variant

In [ ]:
summarize(comparison).round(4)

## Summary per topography (flat / sloped) and variant

In [ ]:
summarize(comparison, by=("topography", "variant")).round(4)

## Largest errors

In [ ]:
subgrid = comparison[comparison["variant"] != "regular_100m"]
subgrid.nlargest(10, "rmse_hmax_m")[["case_name", "split", "variant", "rmse_hmax_m", "csi", "max_hmax_error_m"]]

## Maps: one case per split, plus the worst case

In [ ]:
cases_to_plot = list(case_table.groupby("split").head(1)["case_name"])
cases_to_plot.append(subgrid.nlargest(1, "rmse_hmax_m")["case_name"].iloc[0])
cases_by_name = case_table.set_index("case_name", drop=False)

for case_name in cases_to_plot:
    plot_case(cases_by_name.loc[case_name], PROJECT_ROOT, MODEL_ROOT)

## Largest zsmax errors

In [ ]:
subgrid.nlargest(10, "rmse_zsmax_m")[["case_name", "split", "variant", "rmse_zsmax_m", "bias_zsmax_m",
                                      "p95_abs_zsmax_m", "max_abs_zsmax_m"]]

## zsmax maps: one case per split, plus the worst case

In [ ]:
zsmax_cases = list(case_table.groupby("split").head(1)["case_name"])
zsmax_cases.append(subgrid.nlargest(1, "rmse_zsmax_m")["case_name"].iloc[0])

for case_name in zsmax_cases:
    plot_case(cases_by_name.loc[case_name], PROJECT_ROOT, MODEL_ROOT, quantity="zsmax")